<a href="https://colab.research.google.com/github/Juno-Wong/Dome-41/blob/main/Dome_41.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Domesphere AI-powered Dashboard**

In [3]:
import pandas as pd

# Load all sheets
all_sheets = pd.read_excel("shameless_dashboard_200users.xlsx", sheet_name=None)

# Loop through each sheet and save as CSV
for sheet_name, df in all_sheets.items():
    df.to_csv(f"{sheet_name}.csv", index=False)

# **Clean the dataset**

## **1. Understand the structure**

In [5]:
df_users = pd.read_csv("shameless_users.csv")
df_posts = pd.read_csv("shameless_posts.csv")
df_comments = pd.read_csv("shameless_comments.csv")

In [6]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     200 non-null    object
 1   created_at  200 non-null    object
 2   username    200 non-null    object
 3   city        200 non-null    object
 4   industry    200 non-null    object
 5   gender      200 non-null    object
dtypes: object(6)
memory usage: 9.5+ KB


In [7]:
df_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   post_id              84 non-null     object
 1   user_id              84 non-null     object
 2   post_username        84 non-null     object
 3   post_created_at      84 non-null     object
 4   heading              84 non-null     object
 5   content              84 non-null     object
 6   embed_platform       44 non-null     object
 7   post_like_count      84 non-null     int64 
 8   post_like_usernames  66 non-null     object
 9   comment_count        84 non-null     int64 
 10  latest_comment_at    70 non-null     object
dtypes: int64(2), object(9)
memory usage: 7.3+ KB


In [8]:
df_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   comment_id           191 non-null    object
 1   post_id              191 non-null    object
 2   user_id              191 non-null    object
 3   comment_username     191 non-null    object
 4   comment_created_at   191 non-null    object
 5   content              191 non-null    object
 6   reply_to_comment_id  30 non-null     object
 7   comment_like_count   191 non-null    int64 
dtypes: int64(1), object(7)
memory usage: 12.1+ KB


## **2. Fix Data Types**

Ensures temporal consistency and enables downstream time-based analysis

In [9]:
# Users
df_users["created_at"] = pd.to_datetime(df_users["created_at"])

In [10]:
# Posts
df_posts["post_created_at"] = pd.to_datetime(df_posts["post_created_at"])
df_posts["latest_comment_at"] = pd.to_datetime(df_posts["latest_comment_at"])

In [11]:
# Comments
df_comments["comment_created_at"] = pd.to_datetime(df_comments["comment_created_at"])

### **3. Handle Missing Values (SMART handling only)**

**3.1 Posts**

In [12]:
df_posts["embed_platform"] = df_posts["embed_platform"].fillna("None")

In [13]:
df_posts["post_like_usernames"] = df_posts["post_like_usernames"].fillna("")

In [14]:
df_posts["latest_comment_at"] = df_posts["latest_comment_at"].fillna(df_posts["post_created_at"])

**3.2 Comments**

In [15]:
# Keep as is
# reply_to_comment_id stays NaN

Null values represent top-level comments (not replies).

*Explain: The reply_to_comment_id field contains null values for top-level comments, which do not reply to any existing comment. These null values were preserved, as they represent meaningful structural information rather than missing or erroneous data.*

### **4. Standardise Column Names**

In [16]:
df_users.rename(columns={"created_at": "user_created_at"})

,user_id,user_created_at,username,city,industry,gender
0,92a2012e-738d-4a74-becf-07dd07ab44f7,2026-01-07 08:36:00+00:00,lila,Wellington,Advertising,female
1,ec455a41-4c72-4c7d-9ace-89f42929f39e,2026-01-07 14:12:00+00:00,brookea,London,Student,female
2,03ec60cf-e03a-4715-87b7-3a25540371c5,2026-01-09 19:18:00+00:00,penny66,Bristol,Education,female
3,51efd9cc-9dd5-4483-b6de-ee8e728b82e9,2026-01-10 21:31:00+00:00,paige69,Brisbane,Healthcare,female
4,3c46fc4c-22a4-4dca-9078-1eb0d5d529cc,2026-01-12 09:48:00+00:00,mads,London,Legal,male
...,...,...,...,...,...,...
195,87f6beaf-55e7-47ea-80f0-70cda16730d2,2026-04-01 17:44:00+00:00,alixzz,Melbourne,Student,female
196,ec558623-0f9d-4d8b-88b5-51e03cea6aa4,2026-04-01 17:44:00+00:00,elsieclub,Auckland,Technology,female
197,a4dbd257-6e33-4ccb-bd1e-7ab23ad6c4c8,2026-04-02 21:18:00+00:00,jessie,Singapore,Education,female
198,27ce0655-01cf-4fa8-81e3-021f45b37a31,2026-04-04 07:37:00+00:00,bonnie_x,Christchurch,Advertising,female


Timestamp fields were preserved with distinct naming (e.g., `post_created_at`, `comment_created_at`, `user_created_at`) to maintain semantic clarity, as they represent different events within the system. While schema standardisation was considered, preserving meaningful distinctions was prioritised to support accurate downstream analysis.

### **5. Ensure Key Consistency**

Check relationships -> Remove orphan records and ensure relational integrity.

In [17]:
# Posts must have valid users
df_posts = df_posts[df_posts["user_id"].isin(df_users["user_id"])]

In [18]:
# Comments must have valid posts
df_comments = df_comments[df_comments["post_id"].isin(df_posts["post_id"])]

*Explain: Referential integrity was enforced by ensuring that all foreign key relationships were valid. Specifically, posts were filtered to include only records with `user_id` values present in the users dataset, and comments were filtered to include only valid `post_id` references. This prevents orphan records and ensures consistency across datasets.*

### **6. Remove Duplicates**

In [19]:
df_users = df_users.drop_duplicates()

In [20]:
df_posts = df_posts.drop_duplicates()

In [21]:
df_comments = df_comments.drop_duplicates()

Duplicate records were removed from all datasets to prevent redundancy and ensure data accuracy. This step helps avoid issues such as double counting and inconsistent analysis in downstream processes.

### **7. Clean Text**

In [22]:
import re

def basic_clean(text):
    text = str(text).strip()
    return text

df_posts["content"] = df_posts["content"].apply(basic_clean)
df_comments["content"] = df_comments["content"].apply(basic_clean)

Text fields were standardised by converting all values to string format and removing leading and trailing whitespace. This ensures consistency in textual data and prevents formatting-related issues in downstream processing.

### **8. Sort Data**

In [23]:
df_posts = df_posts.sort_values(by="post_created_at")
df_comments = df_comments.sort_values(by="comment_created_at")

###**9. Cleaned Output**

In [24]:
df_users.to_csv("clean_users.csv", index=False)
df_posts.to_csv("clean_posts.csv", index=False)
df_comments.to_csv("clean_comments.csv", index=False)